In [1]:
import polars as pl
from datetime import timedelta, date, datetime
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
import numpy as np
from tqdm import tqdm
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import itertools

import pickle
from pathlib import Path

# Анализ таргета

In [2]:
# from src.config import RAW_DATA_DIR

# events_path = RAW_DATA_DIR / "dataset/small/marketplace/events"
# #items_path = "D:/HSE_repos/dreamteam-recsys/t_ecd_data/dataset/small/marketplace/items.pq"

In [3]:
events_dir = "../data/raw/T-bank dataset full/T-ECD-small/dataset/small/marketplace/events"

In [4]:
events_df = (
    pl.scan_parquet(f"{events_dir}/*.pq", include_file_paths="path")
    .with_columns(
        pl.col("path")
          .str.extract(r"(\d+)\.pq", group_index=1)
          .cast(pl.Int32)
          .alias("day")
    )
    .sort("day")
    .drop("path")
)

In [5]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32)])

In [6]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,day
duration[μs],u64,str,str,str,str,i32
1082d 17h 39m 51s 751152µs,67838893,"""nfmcg_15106540""","""catalog""","""view""","""android""",1082
1082d 17h 39m 51s 827007µs,89736171,"""nfmcg_12135912""","""u2i""","""view""","""android""",1082
1082d 17h 39m 52s 43343µs,11916495,"""nfmcg_22992567""","""u2i""","""view""","""android""",1082
1082d 17h 39m 52s 87139µs,15515933,"""nfmcg_867469""","""other""","""view""","""android""",1082
1082d 17h 39m 52s 132224µs,49878225,"""nfmcg_6159822""","""u2i""","""view""","""android""",1082


In [7]:
actions_count = events_df.group_by("action_type").agg(pl.len()).collect(engine="streaming")
actions_count.head()

action_type,len
str,u32
"""clickout""",485448
"""like""",41792
"""view""",126147783
"""click""",3772094


Создадим также уменьшенный по масштабу таргет: логарифмированный и квадратичный. Идея: постараться естественным (зависимым от статистики) образом создать таргет, при этом не слишком большим по значению, т.к. огромные редкие таргеты могут заставить модели переобучаться и/или искажать оценки ее качества.

Однако гипотезу нужно будет проверить на практике, потому на текущий момент предлагается 3 варианта: стандартный таргет, логарифмированный и таргет в корне:

In [8]:
actions_count = actions_count.with_columns(
    (pl.col("len").sum() / pl.col("len")).alias("target"),
    (pl.col("len").sum() / pl.col("len")).log1p().alias("log_target"),
    (pl.col("len").sum() / pl.col("len")).sqrt().alias("sqrt_target"),
    
).drop("len")
actions_count

action_type,target,log_target,sqrt_target
str,f64,f64,f64
"""clickout""",268.714913,5.597366,16.392526
"""like""",3121.341812,8.046339,55.86897
"""view""",1.034082,0.710044,1.016898
"""click""",34.582149,3.571844,5.880659


In [9]:
events_df = events_df.join(actions_count.lazy(), on="action_type").with_columns([
    (pl.col("action_type") == "view").cast(pl.Int8).alias("target_view"),
    (pl.col("action_type") == "clickout").cast(pl.Int8).alias("target_clickout"),
    (pl.col("action_type") == "like").cast(pl.Int8).alias("target_like"),
    (pl.col("action_type") == "click").cast(pl.Int8).alias("target_click"),
])

In [10]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,day,target,log_target,sqrt_target,target_view,target_clickout,target_like,target_click
duration[μs],u64,str,str,str,str,i32,f64,f64,f64,i8,i8,i8,i8
1082d 15h 50m 6s 27359µs,33559980,"""nfmcg_3555421""",null,"""like""","""ios""",1082,3121.341812,8.046339,55.86897,0,0,1,0
1082d 15h 55m 44s 274392µs,33559980,"""nfmcg_6043896""",null,"""like""","""ios""",1082,3121.341812,8.046339,55.86897,0,0,1,0
1082d 15h 59m 12s 111031µs,78943612,"""nfmcg_17909747""",null,"""like""","""android""",1082,3121.341812,8.046339,55.86897,0,0,1,0
1082d 15h 59m 52s 902332µs,33559980,"""nfmcg_12799315""",null,"""like""","""ios""",1082,3121.341812,8.046339,55.86897,0,0,1,0
1082d 16h 8m 11s 894056µs,31616293,"""nfmcg_7727120""",null,"""like""","""android""",1082,3121.341812,8.046339,55.86897,0,0,1,0


## Global Temporal Split

In [11]:
def global_temporal_split(
    df: pl.LazyFrame, test_size: int | float = 1, date_column: str = "day"
) -> tuple[pl.LazyFrame, pl.LazyFrame]:
    """
    Разделяет датасет на обучающую и тестовую части на основе глобальной временной границы. 1 день между тестовой и обучающей частью игнорируется.

    Args:
        df: Датасет для разделения
        date_column: Имя столбца с датами
        test_size: Количество дней в тестовой части или доля от общего количества дней

    Returns:
        Кортеж из двух датасетов: обучающая и тестовая части
    """
    min_day, max_day = (
        df.select(
            pl.col(date_column).min().alias("min_day"), pl.col(date_column).max().alias("max_day")
        )
        .collect(engine="streaming")
        .row(0)
    )
    days_all = (max_day - min_day) + 1
    if isinstance(test_size, float):
        test_size = int(days_all * test_size)
        if test_size == 0:
            test_size += 1
        cut_day = max_day - test_size
    else:
        cut_day = max_day - test_size

    if cut_day - 1 < min_day or cut_day + 1 > max_day:
        raise ValueError(
            f"Test size is too large. Test size: {test_size}, min day: {min_day}, max day: {max_day}, cut day: {cut_day}"
        )

    train_df = df.filter(pl.col(date_column) < cut_day)
    test_df = df.filter(pl.col(date_column) > cut_day)

    return train_df, test_df
    

In [12]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32),
        ('target', Float64),
        ('log_target', Float64),
        ('sqrt_target', Float64),
        ('target_view', Int8),
        ('target_clickout', Int8),
        ('target_like', Int8),
        ('target_click', Int8)])

In [13]:
def get_last_k_user_interactions(
    events_df: pl.LazyFrame,
    last_k: int | None = 30,
    date_column: str = "day",
    timestamp_column: str = "timestamp",
    user_column: str = "user_id",
    acceptable_action: list[str] | None = None,
):
    if acceptable_action is None:
        acceptable_action = ["view", "clickout", "like", "click"]
    return (
        events_df.filter(pl.col("action_type").is_in(set(acceptable_action)))
        .group_by(user_column)
        .agg(
            pl.all().sort_by(date_column, timestamp_column).tail(last_k)
            if last_k is not None
            else pl.all().sort_by(date_column, timestamp_column)
        )
    )

In [14]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32),
        ('target', Float64),
        ('log_target', Float64),
        ('sqrt_target', Float64),
        ('target_view', Int8),
        ('target_clickout', Int8),
        ('target_like', Int8),
        ('target_click', Int8)])

In [15]:
train, test = global_temporal_split(events_df, 0.2)

In [16]:
get_last_k_user_interactions(test, acceptable_action=["click", "like", "clickout"]).collect(engine="streaming")

user_id,timestamp,item_id,subdomain,action_type,os,day,target,log_target,sqrt_target,target_view,target_clickout,target_like,target_click
u64,list[duration[μs]],list[str],list[str],list[str],list[str],list[i32],list[f64],list[f64],list[f64],list[i8],list[i8],list[i8],list[i8]
68386380,"[1272d 20h 20m 51s 742910µs, 1276d 9h 48m 10s 910100µs, 1277d 28m 30s 624425µs]","[""nfmcg_21029732"", ""nfmcg_8797298"", ""nfmcg_14081513""]","[""u2i"", ""u2i"", ""u2i""]","[""click"", ""click"", ""click""]","[""android"", ""ios"", ""android""]","[1272, 1276, 1277]","[34.582149, 34.582149, 34.582149]","[3.571844, 3.571844, 3.571844]","[5.880659, 5.880659, 5.880659]","[0, 0, 0]","[0, 0, 0]","[0, 0, 0]","[1, 1, 1]"
3634204,[1289d 17h 2m 48s 300566µs],"[""nfmcg_20987893""]","[""u2i""]","[""click""]","[""ios""]",[1289],[34.582149],[3.571844],[5.880659],[0],[0],[0],[1]
36794173,"[1303d 23h 11m 35s 983945µs, 1304d 15h 39m 12s 830096µs, … 1305d 33m 2s 850775µs]","[""nfmcg_14774260"", ""nfmcg_12174553"", … ""nfmcg_14803421""]","[""catalog"", ""catalog"", … ""catalog""]","[""click"", ""click"", … ""click""]","[""android"", ""other"", … ""other""]","[1303, 1304, … 1305]","[34.582149, 34.582149, … 34.582149]","[3.571844, 3.571844, … 3.571844]","[5.880659, 5.880659, … 5.880659]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 0]","[1, 1, … 1]"
19901219,"[1270d 9h 40m 14s 573204µs, 1270d 14h 27m 39s 598614µs]","[""nfmcg_6755174"", ""nfmcg_2914197""]","[""u2i"", ""u2i""]","[""click"", ""click""]","[""ios"", ""ios""]","[1270, 1270]","[34.582149, 34.582149]","[3.571844, 3.571844]","[5.880659, 5.880659]","[0, 0]","[0, 0]","[0, 0]","[1, 1]"
82933637,"[1281d 9h 37m 49s 662548µs, 1281d 13h 27m 36s 775135µs, … 1295d 17h 20m 16s 275195µs]","[""nfmcg_20987893"", ""nfmcg_27874266"", … ""nfmcg_2307604""]","[""catalog"", ""search"", … ""search""]","[""clickout"", ""click"", … ""click""]","[""android"", ""ios"", … ""android""]","[1281, 1281, … 1295]","[268.714913, 34.582149, … 34.582149]","[5.597366, 3.571844, … 3.571844]","[16.392526, 5.880659, … 5.880659]","[0, 0, … 0]","[1, 0, … 0]","[0, 0, … 0]","[0, 1, … 1]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
38620327,"[1271d 22h 36m 35s 822352µs, 1272d 4h 40m 24s 382381µs, … 1272d 23h 17m 34s 474114µs]","[""nfmcg_1214970"", ""nfmcg_22211380"", … ""nfmcg_22816304""]","[""catalog"", ""i2i"", … ""search""]","[""click"", ""click"", … ""click""]","[""android"", ""ios"", … ""ios""]","[1271, 1272, … 1272]","[34.582149, 34.582149, … 34.582149]","[3.571844, 3.571844, … 3.571844]","[5.880659, 5.880659, … 5.880659]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 0]","[1, 1, … 1]"
21733795,"[1302d 23h 56m 7s 987610µs, 1303d 18h 5m 56s 895607µs]","[""nfmcg_24525513"", ""nfmcg_8631766""]","[""i2i"", ""i2i""]","[""click"", ""click""]","[""android"", ""other""]","[1302, 1303]","[34.582149, 34.582149]","[3.571844, 3.571844]","[5.880659, 5.880659]","[0, 0]","[0, 0]","[0, 0]","[1, 1]"
34740604,"[1294d 22h 46m 51s 125877µs, 1295d 6h 22m 42s 558896µs, … 1295d 9h 46m 32s 405704µs]","[""nfmcg_4104717"", ""nfmcg_4104717"", … ""nfmcg_6136132""]","[""u2i"", ""u2i"", … ""u2i""]","[""clickout"", ""click"", … ""clickout""]","[""android"", ""android"", … ""ios""]","[1294, 1295, … 1295]","[268.714913, 34.582149, … 268.714913]","[5.597366, 3.571844, … 5.597366]","[16.392526, 5.880659, … 16.392526]","[0, 0, … 0]","[1, 0, … 1]","[0, 0, … 0]","[0, 1, … 0]"


In [17]:
def split_cold_start(train_df: pl.LazyFrame, test_df: pl.LazyFrame, user_col: str = "user_id"):
    """
    Split test data into cold-start and non-cold-start subsets by users.

    Parameters
    ----------
    train_df : pl.LazyFrame
        Training data. Used to determine which users are already known to the model.
    test_df : pl.LazyFrame
        Test data that will be split into cold-start and non-cold-start parts.
    user_col : str, optional
        Name of the column containing user identifiers, by default "user_id".

    Returns
    -------
    tuple[pl.LazyFrame, pl.LazyFrame]
        A tuple of two LazyFrames:

        - first element : test subset with cold-start users
          (users present in `test_df` but not in `train_df`);
        - second element : test subset with non-cold-start users
          (users present both in `train_df` and `test_df`).
    """
    cold_start_users = test_df.select(pl.col(user_col).unique()).join(
        train_df.select(pl.col(user_col).unique()), on=user_col, how="anti"
    )
    return test_df.join(cold_start_users, on=user_col), test_df.join(
        cold_start_users, on=user_col, how="anti"
    )

In [18]:
def ndcg_at_k(
    user_based_df: pl.DataFrame,
    relevancy_col: str,
    true_items_col: str,
    predicted_items_col: str,
    predicted_score_col: str,
    top_k: int = 15,
):
    """
    Computes user-based NDCG@k for graded relevance in a recommendation setting.

    Parameters
    ----------
    user_based_df : pl.DataFrame
        Dataframe with user data. Each row must contain user and its lists with: truth
        ground items, their relevancy estimation and model prediction score.
    relevancy_col : str
        Column name contains list of relevancy estimations ground
        truth items (pl.List[float]) for user. Elements order must match `true_items_col`.
    true_items_col : str
        Column name of ground truth items with which user had interactions (pl.List[str]). Relevancy
        of these items must be set in `relevancy_col` respectively. 
    predicted_items_col : str
        Columns name with predicted items (pl.List[str]). Must be set in order matches
        `predicted_score_col`.
    predicted_score_col : str
        Columns name with predicted scores for items in `predicted_items_col` (pl.List[float]).
        Used to sort predictions in descending order.
    top_k : int, optional
        Top k elements to calculate `@k` metric.

    Returns
    -------
    pl.DataFrame
        Columns:
        - ``user_id`` : user identifier;
        - ``ndcg`` : NDCG@k for current user.

    Notes
    -----
    For each user, the function:
    1. Aggregates relevancies for ground-truth items by taking the maximum value for each item.
    2. Joins predicted items with their ground-truth relevancies.
    3. Computes DCG@k using the order induced by the model (sorting by score).
    4. Computes IDCG@k using the ideal order (sorting by ground-truth relevancy).
    5. Returns NDCG@k = DCG@k / IDCG@k, or 0.0 if IDCG@k = 0.
    """
    user_ids = []
    ndcgs = []
    for row in user_based_df.iter_rows(named=True):
        true_items = pl.DataFrame(
            {"truth_items": row[true_items_col], "relevancy": row[relevancy_col]}
        )
        true_items = true_items.group_by("truth_items").agg(
            pl.col("relevancy").max()
        )  # Берем максимальную релевантность для товара
        predictions = (
            pl.DataFrame(
                {"predicted_items": row[predicted_items_col], "score": row[predicted_score_col]}
            )
            .join(
                true_items,
                left_on="predicted_items",
                right_on="truth_items",
                coalesce=True,
                how="left",
            )
            .fill_null(0)
        )
        idcg = (
            predictions.select("relevancy")
            .sort("relevancy", descending=True)
            .head(top_k)
            .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
            .item()
        )
        dcg = (
            predictions.select("score", "relevancy")
            .sort("score", descending=True)
            .head(top_k)
            .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
            .item()
        )
        user_ids.append(row["user_id"])
        ndcgs.append(0.0 if idcg == 0 else dcg / idcg)
    return pl.DataFrame({"user_id": user_ids, "ndcg": ndcgs})

In [19]:
user_based_df = pl.DataFrame({
    "user_id": ["u1", "u2"],
    "true_items": [
        ["A", "B", "C"],   # истиные товары
        ["A", "B", "C"],
    ],
    "relevancy": [
        [3.0, 2.0, 1.0],   # u1: A=3, B=2, C=1
        [3.0, 2.0, 1.0],   # u2: A=3, B=2, C=1
    ],
    "predicted_items": [
        ["A", "B", "C"],   # u1: идеальное ранжирование
        ["C", "B", "A"],   # u2: худшее (перевёрнутое)
    ],
    "predicted_scores": [
        [0.9, 0.8, 0.7],   # по убыванию A>B>C
        [0.9, 0.8, 0.7],   # по убыванию C>B>A
    ],
})

In [20]:
ndcg_at_k(
    user_based_df,
    relevancy_col="relevancy",
    true_items_col="true_items",
    predicted_items_col="predicted_items",
    predicted_score_col="predicted_scores",
    top_k=3,
)

user_id,ndcg
str,f64
"""u1""",1.0
"""u2""",0.789998


## Добавляем метрики Precision@k и Recall@k

Пусть:
- $Rel\_u$ — множество релевантных (истинных) объектов для пользователя $u$ в тесте.
- $Rec\_u@k$ — множество рекомендованных объектов в топ‑k для пользователя $u$.

Тогда для одного пользователя:


$$
Precision@k(u) = \frac{|Rec\_u@k \cap Rel\_u|}{k}
$$


$$
Recall@k(u) = \frac{|Rec\_u@k \cap Rel\_u|}{|Rel\_u|}
$$

Далее берем среднее по всем пользователям:
$$
Precision@k = \frac{1}{N} \sum_{u=1}^{N} Precision@k(u)
$$
$$
Recall@k = \frac{1}{N} \sum_{u=1}^{N} Recall@k(u)
$$

Precision@k показывает качество самого списка рекомендаций. Она говорит о том, сколько "мусора" мы подсовываем пользователю.

Recall@k показывает охват интересов. Она говорит: "Из всего многообразия товаров, которые этот пользователь мог бы полюбить, какую долю мы реально успели ему показать в нашем коротком топе?".

В рекомендательных системах (особенно когда k маленькое, например 10 или 20) Recall обычно важнее, чем Precision. Precision тоже важна, чтобы не раздражать пользователя нерелевантным мусором, но часто мы готовы пожертвовать точностью (показать пару лишних товаров), лишь бы не упустить то, что пользователь реально ищет (сохранить Recall).

In [21]:
def calculate_metrics(df, k):
    """
    Расчет Precision@k и Recall@k
    df: датафрейм с колонками predicted_items и true_items
    k: количество рекомендаций (TOPK)
    """
    # Обрезаем список предсказаний до k элементов
    top_k_preds = pl.col("predicted_items").list.head(k)
    
    # Ищем пересечение обрезанного списка с правдой
    hits_expr = top_k_preds.list.set_intersection(pl.col("true_items")).list.len()
    
    # Вычисляем метрики одной командой select
    metrics = df.select(
        # Precision = (кол-во попаданий / k), берем среднее по всем юзерам
        (hits_expr / k).mean().alias('precision'),
        
        # Recall = (кол-во попаданий / длину реального списка), берем среднее
        (hits_expr / pl.col('true_items').list.len()).mean().alias('recall')
    )
    
    precision_val = metrics['precision'].item()
    recall_val = metrics['recall'].item()
    
    print(f'  Precision@{k}: {precision_val:.6f}')
    print(f'  Recall@{k}:    {recall_val:.6f}')

# Будем реализовывать Truncated SVD

https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html#sklearn.decomposition.TruncatedSVD

>TruncatedSVD implements a variant of singular value decomposition (SVD) that only computes the largest singular values, where is a user-specified parameter.

Источник: https://scikit-learn.org/stable/modules/decomposition.html#lsa

В sklearn.TruncatedSVD используются приближённые алгоритмы SVD. Ищутся k наибольших сингулярных значений и соответствующие им сингулярные векторы, а вклад всех остальных считается пренебрежимо малым и отбрасывается.

Если оставить algorithm='randomized', используется рандомизированный алгоритм Халко (https://arxiv.org/abs/0909.4061) для поиска низкорангового приближения матрицы по первым k сингулярным компонентам. Мы заранее задаём количество компонент n_components = k.

`TruncatedSVD` минимизирует норму Фробениуса разности между исходной матрицей и её приближением.

# Описание бейзлайна

**В данном ноутбуке будем реализовывать Truncated SVD c количеством латентных факторов LATENT_FACTORS = 20 и потом выбирать TOP_K = 15 рекомендаций**

1) Из `train` выбираются столбцы `user_id`, `item_id`, `target` / `sqrt_target` / `log_target` и из них составляется DataFrame `train_data`
2) Фильтруем датафрейм так, чтобы остались только пользователи и айтемы с количеством взаимодейтсвий >= 5
3) Переводим оригинальные id юзеров и айтемов в простые числовые индексы
4) Строим матрицу взаимодействий `R_train_sparse` (User-Item матрица), где по строкам индексы пользователей, по столбцам индексы айтемов, в ячейках - значение таргета. Храним ее в формате хранения csr.
> Если пары (user, item) не было в traindata_indexed, она не попадает в списки индексов и значений, передаваемых в конструктор csr_matrix. В разреженных матрицах (scipy.sparse.csr_matrix) любые координаты, для которых явно не указано значение, по определению считаются равными нулю.

5) Для восстановления тех значений, которые были нулями будем использовать `TruncatedSVD` из `sklearn.decomposition`.

>`TruncatedSVD` (усеченное сингулярное разложение) берет нашу разреженную матрицу взаимодействий (где строки — пользователи, столбцы — >товары) и приближенно раскладывает её в произведение трех матриц:
>
>$$ R \approx U \times \Sigma \times V^T $$
>
>Здесь:
>1. $U$ - показывает отношение пользователей к скрытым факторам.
>2. $\Sigma$ - диагональная матрица с сингулярными числами. Эти числа показывают важность каждого скрытого фактора. Чем больше число, тем больше этот фактор влияет на итоговый выбор.
>3. $V^T$ - показывает, насколько каждый скрытый фактор выражен в конкретном товаре.
>
>
>В `sklearn`:
>
>* матрица `User Features` ($U \cdot \Sigma$) это результат вызова `svd_model.fit_transform(R)`. Каждый пользователь описывается вектором из $k$ чисел (в коде это `LATENT_FACTORS=20`).
>   
>* матрица `Item Features` ($V^T$) это `svd_model.components_`. Каждый товар описывается вектором из $k$ чисел.
>
>Эти 'скрытые факторы' (latent factors) — это неявные характеристики, выявленные алгоритмом.
>
>Когда мы перемножаем полученные матрицы обратно, мы получаем новую плотную матрицу, которая:
>
>* в местах, где были данные старается быть максимально похожей на исходные значения (`target`, `log_target`,`sqrt_target`).
>* в местах, где были нули заполняет их не нулями, а вещественными числами (предсказаниями). Изначально модель стремится предсказать здесь нули (минимизируя ошибку - норму Фробениуса), но из-за использования малого числа компонент ($k$) она вынуждена обобщать паттерны.
>
>Мы сортируем эти предсказанные числа для всех товаров, которые пользователь еще не видел, и рекомендуем те, где предсказанное значение максимально.

## Подготовка данных для построения матрицы взаимодействий

In [22]:
train.head().collect()

timestamp,user_id,item_id,subdomain,action_type,os,day,target,log_target,sqrt_target,target_view,target_clickout,target_like,target_click
duration[μs],u64,str,str,str,str,i32,f64,f64,f64,i8,i8,i8,i8
1082d 89901µs,59328774,"""nfmcg_14777696""","""u2i""","""view""","""android""",1082,1.034082,0.710044,1.016898,1,0,0,0
1082d 163483µs,66295302,"""nfmcg_26098539""","""catalog""","""view""","""android""",1082,1.034082,0.710044,1.016898,1,0,0,0
1082d 864625µs,37542104,"""nfmcg_10840247""","""u2i""","""view""","""android""",1082,1.034082,0.710044,1.016898,1,0,0,0
1082d 889192µs,35193548,"""nfmcg_8040572""","""catalog""","""view""","""android""",1082,1.034082,0.710044,1.016898,1,0,0,0
1082d 936008µs,27256137,"""nfmcg_6770239""","""catalog""","""view""","""android""",1082,1.034082,0.710044,1.016898,1,0,0,0


### Выбираем столбцы 'user_id', 'item_id', 'log_target' из train, это будет датафрейм train_data

In [23]:
train_data = train.select(['user_id', 'item_id', 'log_target']).collect(engine = 'streaming')

In [24]:
train_data.height

101598458

### Оставляем пользователей и айтемы с количеством взаимодействий >= 5. Это нужно для экономии памяти. В результате количество строк в train_data уменьшилось с 101598458 до 98851489

In [25]:
# Для того, чтобы оптимизировать использование памяти оставляем пользователей и айтемы с >= 5 взаимодействиями
MIN_INTERACTIONS = 5

user_counts = train_data.group_by('user_id').agg(pl.len().alias('user_count'))
item_counts = train_data.group_by('item_id').agg(pl.len().alias('item_count'))

In [26]:
user_counts.head()

user_id,user_count
u64,u32
75253539,7
21564242,14
45057051,2
23290806,49
77083397,6


In [27]:
valid_users = user_counts.filter(pl.col('user_count') >= MIN_INTERACTIONS).select('user_id')
valid_items = item_counts.filter(pl.col('item_count') >= MIN_INTERACTIONS).select('item_id')

In [28]:
valid_users.head()

user_id
u64
75253539
21564242
23290806
77083397
73384309


In [29]:
train_data = (
    train_data
    .join(valid_users, on='user_id', how="inner")
    .join(valid_items, on='item_id', how="inner")
)
print(f'Взаимодействий после фильтрации: {train_data.height}')

Взаимодействий после фильтрации: 98851489


### Далее нужно построить матрицу взаимодействий и преобразовать ее в csr-формат

In [30]:
# Создаем таблицу уникальных id с присвоенным индексом (idx)
user_ids_unique = train_data.select(pl.col('user_id')).unique().with_row_index('user_idx') 
item_ids_unique = train_data.select(pl.col('item_id')).unique().with_row_index('item_idx')

In [31]:
user_ids_unique.head()

user_idx,user_id
u32,u64
0,37652843
1,5238637
2,18173851
3,84008768
4,82241013


In [32]:
# Сохраняем размеры матрицы
num_users = user_ids_unique.height
num_items = item_ids_unique.height
print(f"Размерность матрицы: {num_users} пользователей x {num_items} предметов")

Размерность матрицы: 1231601 пользователей x 577509 предметов


In [33]:
# Преобразуем в словарь {'user_idx': [0, 1, ..], 'user_id': [34917205, 2230613, ...]}
user_mapping_dict = user_ids_unique.to_dict(as_series=False)
item_mapping_dict = item_ids_unique.to_dict(as_series=False)

In [34]:
# Преобразование оригинальных id в простые числовые индексы (нужно для матрицы взаимодействий)
user_to_idx = dict(zip(user_mapping_dict['user_id'], user_mapping_dict['user_idx']))
item_to_idx = dict(zip(item_mapping_dict['item_id'], item_mapping_dict['item_idx']))

In [35]:
# Обратный переход 
idx_to_item = {idx: item_id for item_id, idx in item_to_idx.items()}
idx_to_user = {idx: user_id for user_id, idx in user_to_idx.items()}

In [36]:
# Проверка словарей
import itertools

for original_id, index in itertools.islice(user_to_idx.items(), 5):
    print(f'{original_id}: {index}')

37652843: 0
5238637: 1
18173851: 2
84008768: 3
82241013: 4


In [37]:
import itertools

for index, original_id in itertools.islice(idx_to_user.items(), 5):
    print(f'{original_id}: {index}')

37652843: 0
5238637: 1
18173851: 2
84008768: 3
82241013: 4


In [38]:
# Присоединяем созданные индексы (user_idx, item_idx)
train_data_indexed = train_data.join(user_ids_unique, on='user_id', how='left').join(item_ids_unique, on='item_id', how='left')

In [39]:
train_data_indexed.head()

user_id,item_id,log_target,user_idx,item_idx
u64,str,f64,u32,u32
83145641,"""nfmcg_9688964""",3.571844,184899,119333
61804046,"""nfmcg_27615847""",3.571844,1129753,475831
27111017,"""nfmcg_5205476""",3.571844,470075,29041
11786818,"""nfmcg_21388948""",3.571844,1147730,559991
32717788,"""nfmcg_28393071""",3.571844,1211914,337498


In [40]:
# Подготовка данных для CSR матрицы
user_indices = train_data_indexed['user_idx'].to_numpy()
item_indices = train_data_indexed['item_idx'].to_numpy()
relevance_values = train_data_indexed['log_target'].to_numpy()

In [41]:
user_indices

array([ 184899, 1129753,  470075, ...,  570832,   45803,  379390],
      shape=(98851489,), dtype=uint32)

In [42]:
# Создание User-Item матрицы в формате хранения csr
R_train_sparse = csr_matrix(
    (relevance_values, (user_indices, item_indices)),
    shape=(num_users, num_items)
)

print(f'Размер обучающей CSR матрицы: {R_train_sparse.shape}')

Размер обучающей CSR матрицы: (1231601, 577509)


## Обучение TruncatedSVD

In [43]:
LATENT_FACTORS = 20 
svd_model = TruncatedSVD(n_components=LATENT_FACTORS, random_state=42)

# user_features = U * S
user_features = svd_model.fit_transform(R_train_sparse)
# item_features = V^T
item_features = svd_model.components_

## Сохраняем

In [44]:
os.makedirs("model_artifacts", exist_ok=True)

files_to_save = {
    "user_features.pkl": user_features,
    "item_features.pkl": item_features,
    "user_to_idx.pkl": user_to_idx,
    "item_to_idx.pkl": item_to_idx
}

for filename, obj in files_to_save.items():
    with open(f"model_artifacts/{filename}", "wb") as f:
        pickle.dump(obj, f)
    print(f"Файл {filename} успешно сохранен.")

Файл user_features.pkl успешно сохранен.
Файл item_features.pkl успешно сохранен.
Файл user_to_idx.pkl успешно сохранен.
Файл item_to_idx.pkl успешно сохранен.
